# Exploring the Dataset: Ground Truth Labels (All 8 Files)

**Goal:** Understand the structure of the 8 JSONL label files in `labels/` to design the `stg_attack_label_line_raw` table.

All 8 files share an identical JSON schema (`line`, `labels`, `rules`). This notebook consolidates them into a single analysis rather than 8 separate notebooks.

This notebook walks through:
1. Loading all 8 JSONL label files into a unified DataFrame (61,862 rows)
2. Verifying the shared schema (3 fields, no variation)
3. Per-file analysis (label distributions, coverage vs raw log)
4. Cross-file analysis (22 unique labels, 36 unique rules, co-occurrence)
5. Building a raw 1:1 DataFrame (`df_raw`) matching the planned DDL
6. Schema mapping and normalization observations

## 0. Configuration

Set the path to the `russellmitchell/` dataset folder below.

**Default:** Assumes `russellmitchell/` is at the same level as the repo:
```
data-201-group-project/
  data-201-security-log-analysis/   <-- this repo
    notebooks/                       <-- this notebook is here
  russellmitchell/                   <-- dataset is here
```

In [1]:
from pathlib import Path

# --- CHANGE THIS if your dataset is in a different location ---
DATASET_ROOT = Path("..") / ".." / "russellmitchell"
LABELS_ROOT = DATASET_ROOT / "labels"
GATHER_ROOT = DATASET_ROOT / "gather"

# All 8 label files with metadata
LABEL_FILES = [
    {
        "host": "inet-firewall",
        "log": "dnsmasq.log",
        "label_path": LABELS_ROOT / "inet-firewall" / "logs" / "dnsmasq.log",
        "raw_path": GATHER_ROOT / "inet-firewall" / "logs" / "dnsmasq.log",
        "raw_lines": 275_900,
    },
    {
        "host": "intranet_server",
        "log": "access.log.2",
        "label_path": LABELS_ROOT
        / "intranet_server"
        / "logs"
        / "apache2"
        / "intranet.smith.russellmitchell.com-access.log.2",
        "raw_path": GATHER_ROOT
        / "intranet_server"
        / "logs"
        / "apache2"
        / "intranet.smith.russellmitchell.com-access.log.2",
        "raw_lines": 8_530,
    },
    {
        "host": "intranet_server",
        "log": "error.log.2",
        "label_path": LABELS_ROOT
        / "intranet_server"
        / "logs"
        / "apache2"
        / "intranet.smith.russellmitchell.com-error.log.2",
        "raw_path": GATHER_ROOT
        / "intranet_server"
        / "logs"
        / "apache2"
        / "intranet.smith.russellmitchell.com-error.log.2",
        "raw_lines": 36,
    },
    {
        "host": "intranet_server",
        "log": "audit.log",
        "label_path": LABELS_ROOT / "intranet_server" / "logs" / "audit" / "audit.log",
        "raw_path": GATHER_ROOT / "intranet_server" / "logs" / "audit" / "audit.log",
        "raw_lines": 2_316,
    },
    {
        "host": "intranet_server",
        "log": "auth.log",
        "label_path": LABELS_ROOT / "intranet_server" / "logs" / "auth.log",
        "raw_path": GATHER_ROOT / "intranet_server" / "logs" / "auth.log",
        "raw_lines": 272,
    },
    {
        "host": "monitoring",
        "log": "cpu.log",
        "label_path": LABELS_ROOT
        / "monitoring"
        / "logs"
        / "logstash"
        / "intranet-server"
        / "2022-01-24-system.cpu.log",
        "raw_path": GATHER_ROOT
        / "monitoring"
        / "logs"
        / "logstash"
        / "intranet-server"
        / "2022-01-24-system.cpu.log",
        "raw_lines": 1_920,
    },
    {
        "host": "vpn",
        "log": "openvpn.log",
        "label_path": LABELS_ROOT / "vpn" / "logs" / "openvpn.log",
        "raw_path": GATHER_ROOT / "vpn" / "logs" / "openvpn.log",
        "raw_lines": 5_537,
    },
    {
        "host": "internal_share",
        "log": "audit.log",
        "label_path": LABELS_ROOT / "internal_share" / "logs" / "audit" / "audit.log",
        "raw_path": GATHER_ROOT / "internal_share" / "logs" / "audit" / "audit.log",
        "raw_lines": 732,
    },
]

# Verify all files exist
for entry in LABEL_FILES:
    label_status = "FOUND" if entry["label_path"].exists() else "MISSING"
    raw_status = "FOUND" if entry["raw_path"].exists() else "MISSING"
    print(f"  {entry['host']}/{entry['log']}: labels={label_status}, raw={raw_status}")

  inet-firewall/dnsmasq.log: labels=FOUND, raw=FOUND
  intranet_server/access.log.2: labels=FOUND, raw=FOUND
  intranet_server/error.log.2: labels=FOUND, raw=FOUND
  intranet_server/audit.log: labels=FOUND, raw=FOUND
  intranet_server/auth.log: labels=FOUND, raw=FOUND
  monitoring/cpu.log: labels=FOUND, raw=FOUND
  vpn/openvpn.log: labels=FOUND, raw=FOUND
  internal_share/audit.log: labels=FOUND, raw=FOUND


## 1. Load All Label Files

Each JSONL file has one JSON object per line:
```json
{"line": <int>, "labels": [<str>, ...], "rules": {<label>: [<rule>, ...], ...}}
```

We load all 8 files, tag each row with its source host and log, then combine into a single DataFrame.

In [2]:
import json

import pandas as pd

all_records = []

for entry in LABEL_FILES:
    with open(entry["label_path"]) as fh:
        for raw_line in fh:
            record = json.loads(raw_line)
            record["source_host"] = entry["host"]
            record["source_log"] = entry["log"]
            record["raw_lines_total"] = entry["raw_lines"]
            all_records.append(record)

df_all = pd.DataFrame(all_records)
print(f"Total records loaded: {len(df_all)}")
print(f"Columns: {list(df_all.columns)}")
print(f"Shape: {df_all.shape}")
print()

# Per-file counts
file_counts = df_all.groupby(["source_host", "source_log"]).size().reset_index(name="count")
print("Per-file record counts:")
print(file_counts.to_string(index=False))
print(f"\nTotal: {file_counts['count'].sum()}")

Total records loaded: 61862
Columns: ['line', 'labels', 'rules', 'source_host', 'source_log', 'raw_lines_total']
Shape: (61862, 6)

Per-file record counts:
    source_host   source_log  count
  inet-firewall  dnsmasq.log  54035
 internal_share    audit.log      2
intranet_server access.log.2   7695
intranet_server    audit.log      9
intranet_server     auth.log      8
intranet_server  error.log.2     36
     monitoring      cpu.log     49
            vpn  openvpn.log     28

Total: 61862


## 2. Schema Inventory

Confirm all 8 files share the identical JSON keys and data types.

In [3]:
# Check that every record has exactly the same JSON keys
json_keys_per_file = {}
for entry in LABEL_FILES:
    key = f"{entry['host']}/{entry['log']}"
    with open(entry["label_path"]) as fh:
        first_line = json.loads(fh.readline())
        json_keys_per_file[key] = set(first_line.keys())

# Compare all key sets
all_key_sets = list(json_keys_per_file.values())
schema_uniform = all(ks == all_key_sets[0] for ks in all_key_sets)
print(f"Schema uniform across all 8 files: {schema_uniform}")
print(f"JSON keys: {sorted(all_key_sets[0])}")

Schema uniform across all 8 files: True
JSON keys: ['labels', 'line', 'rules']


In [4]:
# Verify rules keys always match labels entries (no orphan rules)
mismatches = 0
for _, row in df_all.iterrows():
    label_set = set(row["labels"])
    rules_keys = set(row["rules"].keys())
    if label_set != rules_keys:
        mismatches += 1
        print(
            f"  MISMATCH at {row['source_host']}/{row['source_log']} line {row['line']}: "
            f"labels={label_set}, rules_keys={rules_keys}"
        )

print(f"Rules-labels mismatches: {mismatches}")

Rules-labels mismatches: 0


In [5]:
# Field type verification
print("=== Field types ===")
print(
    f"  line: {type(df_all['line'].iloc[0]).__name__} (range: {df_all['line'].min()} - {df_all['line'].max()})"
)
print(f"  labels: {type(df_all['labels'].iloc[0]).__name__} (list of str)")
print(f"  rules: {type(df_all['rules'].iloc[0]).__name__} (dict of str -> list[str])")
print()

# Labels per line distribution
labels_per_line = df_all["labels"].apply(len)
print("=== Labels per line ===")
print(f"  min: {labels_per_line.min()}")
print(f"  max: {labels_per_line.max()}")
print(f"  mean: {labels_per_line.mean():.2f}")
print("  distribution:")
for n_labels, count in sorted(labels_per_line.value_counts().items()):
    print(f"    {n_labels} labels: {count:,} lines ({count / len(df_all) * 100:.1f}%)")

print()

# Rules per label distribution
rules_counts = []
for _, row in df_all.iterrows():
    for _lbl_name, rule_list in row["rules"].items():
        rules_counts.append(len(rule_list))
print("=== Rules per label ===")
print(f"  min: {min(rules_counts)}")
print(f"  max: {max(rules_counts)}")

=== Field types ===
  line: int64 (range: 1 - 254393)
  labels: list (list of str)
  rules: dict (dict of str -> list[str])

=== Labels per line ===
  min: 2
  max: 4
  mean: 2.98
  distribution:
    2 labels: 1,073 lines (1.7%)
    3 labels: 60,785 lines (98.3%)
    4 labels: 4 lines (0.0%)

=== Rules per label ===
  min: 1
  max: 3


## 3. Per-File Analysis

One subsection per label file. For each: line range, label distribution, combo distribution, rule distribution, and coverage vs raw log.

### 3.1 inet-firewall/dnsmasq.log

In [6]:
from collections import Counter

df_file = df_all[
    (df_all["source_host"] == "inet-firewall") & (df_all["source_log"] == "dnsmasq.log")
].copy()
raw_total = 275900

print("=== 3.1 inet-firewall/dnsmasq.log ===")
print(f"Labeled lines: {len(df_file)}")
print(f"Raw log lines: {raw_total}")
print(f"Coverage: {len(df_file)}/{raw_total} = {len(df_file) / raw_total * 100:.1f}%")
print(f"Line range: {df_file['line'].min()} - {df_file['line'].max()}")
print()

# Label distribution
label_ctr = Counter()
for label_list in df_file["labels"]:
    for lbl_name in label_list:
        label_ctr[lbl_name] += 1
print(f"Unique labels: {len(label_ctr)}")
print("Label distribution:")
for lbl_name, cnt in label_ctr.most_common():
    print(f"  {lbl_name}: {cnt}")

print()

# Label combo distribution
combo_ctr = Counter()
for label_list in df_file["labels"]:
    combo_ctr[tuple(sorted(label_list))] += 1
print(f"Unique label combos: {len(combo_ctr)}")
print("Label combo distribution:")
for combo, cnt in combo_ctr.most_common():
    print(f"  {combo}: {cnt}")

print()

# Rule distribution
rule_ctr = Counter()
for rules_dict in df_file["rules"]:
    for rule_list in rules_dict.values():
        for rule_name in rule_list:
            rule_ctr[rule_name] += 1
print(f"Unique rules: {len(rule_ctr)}")
print("Rule distribution:")
for rule_name, cnt in rule_ctr.most_common():
    print(f"  {rule_name}: {cnt}")

=== 3.1 inet-firewall/dnsmasq.log ===
Labeled lines: 54035
Raw log lines: 275900
Coverage: 54035/275900 = 19.6%
Line range: 1 - 254393

Unique labels: 13
Label distribution:
  dnsteal: 53054
  attacker: 53054
  dnsteal-received: 53006
  foothold: 969
  service_scan: 443
  dns_scan: 414
  network_scan: 92
  dnsteal-dropped: 48
  webshell_cmd: 12
  escalate: 12
  dirb: 8
  wpscan: 8
  traceroute: 4

Unique label combos: 9
Label combo distribution:
  ('attacker', 'dnsteal', 'dnsteal-received'): 53006
  ('foothold', 'service_scan'): 443
  ('dns_scan', 'foothold'): 414
  ('foothold', 'network_scan'): 92
  ('attacker', 'dnsteal', 'dnsteal-dropped'): 48
  ('escalate', 'webshell_cmd'): 12
  ('dirb', 'foothold'): 8
  ('foothold', 'wpscan'): 8
  ('foothold', 'traceroute'): 4

Unique rules: 16
Rule distribution:
  dnsteal.domain.match: 106108
  dnsteal.domain.received: 53006
  attacker_foothold_dnsmasq_service_scan_sub_query: 654
  attacker_foothold_dnsmasq_dns_brute_scan_sub_query: 558
  attacke

### 3.2 intranet_server/access.log.2

In [7]:
from collections import Counter

df_file = df_all[
    (df_all["source_host"] == "intranet_server") & (df_all["source_log"] == "access.log.2")
].copy()
raw_total = 8530

print("=== 3.2 intranet_server/access.log.2 ===")
print(f"Labeled lines: {len(df_file)}")
print(f"Raw log lines: {raw_total}")
print(f"Coverage: {len(df_file)}/{raw_total} = {len(df_file) / raw_total * 100:.1f}%")
print(f"Line range: {df_file['line'].min()} - {df_file['line'].max()}")
print()

# Label distribution
label_ctr = Counter()
for label_list in df_file["labels"]:
    for lbl_name in label_list:
        label_ctr[lbl_name] += 1
print(f"Unique labels: {len(label_ctr)}")
print("Label distribution:")
for lbl_name, cnt in label_ctr.most_common():
    print(f"  {lbl_name}: {cnt}")

print()

# Label combo distribution
combo_ctr = Counter()
for label_list in df_file["labels"]:
    combo_ctr[tuple(sorted(label_list))] += 1
print(f"Unique label combos: {len(combo_ctr)}")
print("Label combo distribution:")
for combo, cnt in combo_ctr.most_common():
    print(f"  {combo}: {cnt}")

print()

# Rule distribution
rule_ctr = Counter()
for rules_dict in df_file["rules"]:
    for rule_list in rules_dict.values():
        for rule_name in rule_list:
            rule_ctr[rule_name] += 1
print(f"Unique rules: {len(rule_ctr)}")
print("Rule distribution:")
for rule_name, cnt in rule_ctr.most_common():
    print(f"  {rule_name}: {cnt}")

=== 3.2 intranet_server/access.log.2 ===
Labeled lines: 7695
Raw log lines: 8530
Coverage: 7695/8530 = 90.2%
Line range: 832 - 8529

Unique labels: 8
Label distribution:
  foothold: 7691
  attacker_http: 7687
  dirb: 4462
  wpscan: 3186
  webshell_cmd: 32
  service_scan: 12
  escalate: 4
  webshell_upload: 3

Unique label combos: 7
Label combo distribution:
  ('attacker_http', 'dirb', 'foothold'): 4462
  ('attacker_http', 'foothold', 'wpscan'): 3186
  ('attacker_http', 'foothold', 'webshell_cmd'): 28
  ('attacker_http', 'foothold', 'service_scan'): 8
  ('foothold', 'service_scan'): 4
  ('escalate', 'webshell_cmd'): 4
  ('attacker_http', 'foothold', 'webshell_upload'): 3

Unique rules: 8
Rule distribution:
  attacker.foothold.apache.access: 15374
  attacker.dirb.time: 4462
  attacker.wpscan.time: 3186
  attacker.foothold.apache.access_error: 48
  attacker.escalate.webshell.cmd.http: 28
  attacker.service_scan: 24
  attacker.escalate.webshell.cmd.http_prepare_crack: 8
  attacker.webshell

### 3.3 intranet_server/error.log.2

In [8]:
from collections import Counter

df_file = df_all[
    (df_all["source_host"] == "intranet_server") & (df_all["source_log"] == "error.log.2")
].copy()
raw_total = 36

print("=== 3.3 intranet_server/error.log.2 ===")
print(f"Labeled lines: {len(df_file)}")
print(f"Raw log lines: {raw_total}")
print(f"Coverage: {len(df_file)}/{raw_total} = {len(df_file) / raw_total * 100:.1f}%")
print(f"Line range: {df_file['line'].min()} - {df_file['line'].max()}")
print()

# Label distribution
label_ctr = Counter()
for label_list in df_file["labels"]:
    for lbl_name in label_list:
        label_ctr[lbl_name] += 1
print(f"Unique labels: {len(label_ctr)}")
print("Label distribution:")
for lbl_name, cnt in label_ctr.most_common():
    print(f"  {lbl_name}: {cnt}")

print()

# Label combo distribution
combo_ctr = Counter()
for label_list in df_file["labels"]:
    combo_ctr[tuple(sorted(label_list))] += 1
print(f"Unique label combos: {len(combo_ctr)}")
print("Label combo distribution:")
for combo, cnt in combo_ctr.most_common():
    print(f"  {combo}: {cnt}")

print()

# Rule distribution
rule_ctr = Counter()
for rules_dict in df_file["rules"]:
    for rule_list in rules_dict.values():
        for rule_name in rule_list:
            rule_ctr[rule_name] += 1
print(f"Unique rules: {len(rule_ctr)}")
print("Rule distribution:")
for rule_name, cnt in rule_ctr.most_common():
    print(f"  {rule_name}: {cnt}")

=== 3.3 intranet_server/error.log.2 ===
Labeled lines: 36
Raw log lines: 36
Coverage: 36/36 = 100.0%
Line range: 1 - 36

Unique labels: 4
Label distribution:
  attacker_http: 36
  foothold: 36
  dirb: 23
  wpscan: 13

Unique label combos: 2
Label combo distribution:
  ('attacker_http', 'dirb', 'foothold'): 23
  ('attacker_http', 'foothold', 'wpscan'): 13

Unique rules: 5
Rule distribution:
  attacker.foothold.apache.error: 48
  attacker.foothold.apache.access_error: 48
  attacker.foothold.apache.error_substring: 24
  attacker.dirb.time: 23
  attacker.wpscan.time: 13


### 3.4 intranet_server/audit.log

In [9]:
from collections import Counter

df_file = df_all[
    (df_all["source_host"] == "intranet_server") & (df_all["source_log"] == "audit.log")
].copy()
raw_total = 2316

print("=== 3.4 intranet_server/audit.log ===")
print(f"Labeled lines: {len(df_file)}")
print(f"Raw log lines: {raw_total}")
print(f"Coverage: {len(df_file)}/{raw_total} = {len(df_file) / raw_total * 100:.1f}%")
print(f"Line range: {df_file['line'].min()} - {df_file['line'].max()}")
print()

# Label distribution
label_ctr = Counter()
for label_list in df_file["labels"]:
    for lbl_name in label_list:
        label_ctr[lbl_name] += 1
print(f"Unique labels: {len(label_ctr)}")
print("Label distribution:")
for lbl_name, cnt in label_ctr.most_common():
    print(f"  {lbl_name}: {cnt}")

print()

# Label combo distribution
combo_ctr = Counter()
for label_list in df_file["labels"]:
    combo_ctr[tuple(sorted(label_list))] += 1
print(f"Unique label combos: {len(combo_ctr)}")
print("Label combo distribution:")
for combo, cnt in combo_ctr.most_common():
    print(f"  {combo}: {cnt}")

print()

# Rule distribution
rule_ctr = Counter()
for rules_dict in df_file["rules"]:
    for rule_list in rules_dict.values():
        for rule_name in rule_list:
            rule_ctr[rule_name] += 1
print(f"Unique rules: {len(rule_ctr)}")
print("Rule distribution:")
for rule_name, cnt in rule_ctr.most_common():
    print(f"  {rule_name}: {cnt}")

=== 3.4 intranet_server/audit.log ===
Labeled lines: 9
Raw log lines: 2316
Coverage: 9/2316 = 0.4%
Line range: 1860 - 1868

Unique labels: 4
Label distribution:
  escalate: 9
  escalated_command: 5
  escalated_sudo_command: 5
  attacker_change_user: 4

Unique label combos: 2
Label combo distribution:
  ('escalate', 'escalated_command', 'escalated_sudo_command'): 5
  ('attacker_change_user', 'escalate'): 4

Unique rules: 3
Rule distribution:
  attacker.escalate.audit.sudo.command.events: 15
  attacker.escalate.audit.su.login: 8
  attacker.escalate.audit.sudo.command.start: 3


### 3.5 intranet_server/auth.log

In [10]:
from collections import Counter

df_file = df_all[
    (df_all["source_host"] == "intranet_server") & (df_all["source_log"] == "auth.log")
].copy()
raw_total = 272

print("=== 3.5 intranet_server/auth.log ===")
print(f"Labeled lines: {len(df_file)}")
print(f"Raw log lines: {raw_total}")
print(f"Coverage: {len(df_file)}/{raw_total} = {len(df_file) / raw_total * 100:.1f}%")
print(f"Line range: {df_file['line'].min()} - {df_file['line'].max()}")
print()

# Label distribution
label_ctr = Counter()
for label_list in df_file["labels"]:
    for lbl_name in label_list:
        label_ctr[lbl_name] += 1
print(f"Unique labels: {len(label_ctr)}")
print("Label distribution:")
for lbl_name, cnt in label_ctr.most_common():
    print(f"  {lbl_name}: {cnt}")

print()

# Label combo distribution
combo_ctr = Counter()
for label_list in df_file["labels"]:
    combo_ctr[tuple(sorted(label_list))] += 1
print(f"Unique label combos: {len(combo_ctr)}")
print("Label combo distribution:")
for combo, cnt in combo_ctr.most_common():
    print(f"  {combo}: {cnt}")

print()

# Rule distribution
rule_ctr = Counter()
for rules_dict in df_file["rules"]:
    for rule_list in rules_dict.values():
        for rule_name in rule_list:
            rule_ctr[rule_name] += 1
print(f"Unique rules: {len(rule_ctr)}")
print("Rule distribution:")
for rule_name, cnt in rule_ctr.most_common():
    print(f"  {rule_name}: {cnt}")

=== 3.5 intranet_server/auth.log ===
Labeled lines: 8
Raw log lines: 272
Coverage: 8/272 = 2.9%
Line range: 145 - 152

Unique labels: 5
Label distribution:
  escalate: 8
  escalated_command: 5
  escalated_sudo_command: 5
  attacker_change_user: 4
  escalated_sudo_session: 3

Unique label combos: 4
Label combo distribution:
  ('attacker_change_user', 'escalate'): 3
  ('escalate', 'escalated_command', 'escalated_sudo_command', 'escalated_sudo_session'): 3
  ('attacker_change_user', 'escalate', 'escalated_command', 'escalated_sudo_command'): 1
  ('escalate', 'escalated_command', 'escalated_sudo_command'): 1

Unique rules: 5
Rule distribution:
  attacker.escalate.sudo.open: 12
  attacker.escalate.sudo.command: 9
  attacker.escalate.su.login: 6
  attacker.escalate.systemd.newsession.after: 4
  attacker.escalate.audit.sudo.command.start: 3


### 3.6 monitoring/cpu.log

In [11]:
from collections import Counter

df_file = df_all[
    (df_all["source_host"] == "monitoring") & (df_all["source_log"] == "cpu.log")
].copy()
raw_total = 1920

print("=== 3.6 monitoring/cpu.log ===")
print(f"Labeled lines: {len(df_file)}")
print(f"Raw log lines: {raw_total}")
print(f"Coverage: {len(df_file)}/{raw_total} = {len(df_file) / raw_total * 100:.1f}%")
print(f"Line range: {df_file['line'].min()} - {df_file['line'].max()}")
print()

# Label distribution
label_ctr = Counter()
for label_list in df_file["labels"]:
    for lbl_name in label_list:
        label_ctr[lbl_name] += 1
print(f"Unique labels: {len(label_ctr)}")
print("Label distribution:")
for lbl_name, cnt in label_ctr.most_common():
    print(f"  {lbl_name}: {cnt}")

print()

# Label combo distribution
combo_ctr = Counter()
for label_list in df_file["labels"]:
    combo_ctr[tuple(sorted(label_list))] += 1
print(f"Unique label combos: {len(combo_ctr)}")
print("Label combo distribution:")
for combo, cnt in combo_ctr.most_common():
    print(f"  {combo}: {cnt}")

print()

# Rule distribution
rule_ctr = Counter()
for rules_dict in df_file["rules"]:
    for rule_list in rules_dict.values():
        for rule_name in rule_list:
            rule_ctr[rule_name] += 1
print(f"Unique rules: {len(rule_ctr)}")
print("Rule distribution:")
for rule_name, cnt in rule_ctr.most_common():
    print(f"  {rule_name}: {cnt}")

=== 3.6 monitoring/cpu.log ===
Labeled lines: 49
Raw log lines: 1920
Coverage: 49/1920 = 2.6%
Line range: 321 - 369

Unique labels: 2
Label distribution:
  escalate: 49
  crack_passwords: 49

Unique label combos: 1
Label combo distribution:
  ('crack_passwords', 'escalate'): 49

Unique rules: 1
Rule distribution:
  attacker.escalate.wpcrack: 98


### 3.7 vpn/openvpn.log

In [12]:
from collections import Counter

df_file = df_all[(df_all["source_host"] == "vpn") & (df_all["source_log"] == "openvpn.log")].copy()
raw_total = 5537

print("=== 3.7 vpn/openvpn.log ===")
print(f"Labeled lines: {len(df_file)}")
print(f"Raw log lines: {raw_total}")
print(f"Coverage: {len(df_file)}/{raw_total} = {len(df_file) / raw_total * 100:.1f}%")
print(f"Line range: {df_file['line'].min()} - {df_file['line'].max()}")
print()

# Label distribution
label_ctr = Counter()
for label_list in df_file["labels"]:
    for lbl_name in label_list:
        label_ctr[lbl_name] += 1
print(f"Unique labels: {len(label_ctr)}")
print("Label distribution:")
for lbl_name, cnt in label_ctr.most_common():
    print(f"  {lbl_name}: {cnt}")

print()

# Label combo distribution
combo_ctr = Counter()
for label_list in df_file["labels"]:
    combo_ctr[tuple(sorted(label_list))] += 1
print(f"Unique label combos: {len(combo_ctr)}")
print("Label combo distribution:")
for combo, cnt in combo_ctr.most_common():
    print(f"  {combo}: {cnt}")

print()

# Rule distribution
rule_ctr = Counter()
for rules_dict in df_file["rules"]:
    for rule_list in rules_dict.values():
        for rule_name in rule_list:
            rule_ctr[rule_name] += 1
print(f"Unique rules: {len(rule_ctr)}")
print("Rule distribution:")
for rule_name, cnt in rule_ctr.most_common():
    print(f"  {rule_name}: {cnt}")

=== 3.7 vpn/openvpn.log ===
Labeled lines: 28
Raw log lines: 5537
Coverage: 28/5537 = 0.5%
Line range: 4331 - 4358

Unique labels: 2
Label distribution:
  attacker_vpn: 28
  foothold: 28

Unique label combos: 1
Label combo distribution:
  ('attacker_vpn', 'foothold'): 28

Unique rules: 1
Rule distribution:
  attacker.foothold.vpn.ip: 56


### 3.8 internal_share/audit.log

In [13]:
from collections import Counter

df_file = df_all[
    (df_all["source_host"] == "internal_share") & (df_all["source_log"] == "audit.log")
].copy()
raw_total = 732

print("=== 3.8 internal_share/audit.log ===")
print(f"Labeled lines: {len(df_file)}")
print(f"Raw log lines: {raw_total}")
print(f"Coverage: {len(df_file)}/{raw_total} = {len(df_file) / raw_total * 100:.1f}%")
print(f"Line range: {df_file['line'].min()} - {df_file['line'].max()}")
print()

# Label distribution
label_ctr = Counter()
for label_list in df_file["labels"]:
    for lbl_name in label_list:
        label_ctr[lbl_name] += 1
print(f"Unique labels: {len(label_ctr)}")
print("Label distribution:")
for lbl_name, cnt in label_ctr.most_common():
    print(f"  {lbl_name}: {cnt}")

print()

# Label combo distribution
combo_ctr = Counter()
for label_list in df_file["labels"]:
    combo_ctr[tuple(sorted(label_list))] += 1
print(f"Unique label combos: {len(combo_ctr)}")
print("Label combo distribution:")
for combo, cnt in combo_ctr.most_common():
    print(f"  {combo}: {cnt}")

print()

# Rule distribution
rule_ctr = Counter()
for rules_dict in df_file["rules"]:
    for rule_list in rules_dict.values():
        for rule_name in rule_list:
            rule_ctr[rule_name] += 1
print(f"Unique rules: {len(rule_ctr)}")
print("Rule distribution:")
for rule_name, cnt in rule_ctr.most_common():
    print(f"  {rule_name}: {cnt}")

=== 3.8 internal_share/audit.log ===
Labeled lines: 2
Raw log lines: 732
Coverage: 2/732 = 0.3%
Line range: 667 - 668

Unique labels: 3
Label distribution:
  dnsteal: 2
  exfiltration-service: 2
  attacker: 2

Unique label combos: 1
Label combo distribution:
  ('attacker', 'dnsteal', 'exfiltration-service'): 2

Unique rules: 1
Rule distribution:
  exfil.service: 6


## 4. Cross-File Analysis

Aggregate analysis across all 8 files: label frequency, co-occurrence, attack phase mapping, rule inventory.

### 4.1 Label Frequency (All Files)

In [14]:
from collections import Counter

# Global label frequency
global_label_ctr = Counter()
for label_list in df_all["labels"]:
    for lbl_name in label_list:
        global_label_ctr[lbl_name] += 1

print(f"=== Global Label Frequency ({len(global_label_ctr)} unique labels) ===")
print(f"{'Label':<30} {'Count':>8} {'%':>7}")
print("-" * 47)
total_label_occurrences = sum(global_label_ctr.values())
for lbl_name, cnt in global_label_ctr.most_common():
    print(f"{lbl_name:<30} {cnt:>8,} {cnt / total_label_occurrences * 100:>6.1f}%")
print("-" * 47)
print(f"{'Total occurrences':<30} {total_label_occurrences:>8,}")

=== Global Label Frequency (22 unique labels) ===
Label                             Count       %
-----------------------------------------------
dnsteal                          53,056   28.8%
attacker                         53,056   28.8%
dnsteal-received                 53,006   28.7%
foothold                          8,724    4.7%
attacker_http                     7,723    4.2%
dirb                              4,493    2.4%
wpscan                            3,207    1.7%
service_scan                        455    0.2%
dns_scan                            414    0.2%
network_scan                         92    0.0%
escalate                             82    0.0%
crack_passwords                      49    0.0%
dnsteal-dropped                      48    0.0%
webshell_cmd                         44    0.0%
attacker_vpn                         28    0.0%
escalated_command                    10    0.0%
escalated_sudo_command               10    0.0%
attacker_change_user                  

### 4.2 Label Co-Occurrence

In [15]:
# Label combo frequency across all files
global_combo_ctr = Counter()
for label_list in df_all["labels"]:
    global_combo_ctr[tuple(sorted(label_list))] += 1

print(f"=== Label Co-Occurrence ({len(global_combo_ctr)} unique combos) ===")
print(f"{'Combo':<65} {'Count':>8}")
print("-" * 75)
for combo, cnt in global_combo_ctr.most_common():
    combo_str = " + ".join(combo)
    print(f"{combo_str:<65} {cnt:>8,}")

=== Label Co-Occurrence (21 unique combos) ===
Combo                                                                Count
---------------------------------------------------------------------------
attacker + dnsteal + dnsteal-received                               53,006
attacker_http + dirb + foothold                                      4,485
attacker_http + foothold + wpscan                                    3,199
foothold + service_scan                                                447
dns_scan + foothold                                                    414
foothold + network_scan                                                 92
crack_passwords + escalate                                              49
attacker + dnsteal + dnsteal-dropped                                    48
attacker_http + foothold + webshell_cmd                                 28
attacker_vpn + foothold                                                 28
escalate + webshell_cmd                             

In [16]:
# Co-occurrence matrix: which labels appear together
all_labels_sorted = sorted(global_label_ctr.keys())
cooccurrence = pd.DataFrame(0, index=all_labels_sorted, columns=all_labels_sorted)

for label_list in df_all["labels"]:
    labels_sorted = sorted(set(label_list))
    for i, lbl_a in enumerate(labels_sorted):
        for lbl_b in labels_sorted[i:]:
            cooccurrence.loc[lbl_a, lbl_b] += 1
            if lbl_a != lbl_b:
                cooccurrence.loc[lbl_b, lbl_a] += 1

# Show only non-zero off-diagonal pairs
print("=== Co-occurrence pairs (top 20) ===")
pairs = []
for i, lbl_a in enumerate(all_labels_sorted):
    for lbl_b in all_labels_sorted[i + 1 :]:
        if cooccurrence.loc[lbl_a, lbl_b] > 0:
            pairs.append((lbl_a, lbl_b, cooccurrence.loc[lbl_a, lbl_b]))
pairs.sort(key=lambda x: x[2], reverse=True)
for lbl_a, lbl_b, cnt in pairs[:20]:
    print(f"  {lbl_a} + {lbl_b}: {cnt:,}")

=== Co-occurrence pairs (top 20) ===
  attacker + dnsteal: 53,056
  attacker + dnsteal-received: 53,006
  dnsteal + dnsteal-received: 53,006
  attacker_http + foothold: 7,723
  dirb + foothold: 4,493
  attacker_http + dirb: 4,485
  foothold + wpscan: 3,207
  attacker_http + wpscan: 3,199
  foothold + service_scan: 455
  dns_scan + foothold: 414
  foothold + network_scan: 92
  crack_passwords + escalate: 49
  attacker + dnsteal-dropped: 48
  dnsteal + dnsteal-dropped: 48
  attacker_http + webshell_cmd: 28
  attacker_vpn + foothold: 28
  foothold + webshell_cmd: 28
  escalate + webshell_cmd: 16
  escalate + escalated_command: 10
  escalate + escalated_sudo_command: 10


### 4.3 Attack Phase Grouping

In [17]:
# Map 22 labels to attack phases (from data_scope_and_findings.md taxonomy)
LABEL_TO_PHASE = {
    # Initial Access
    "attacker_vpn": "Initial Access",
    "foothold": "Initial Access",
    # Reconnaissance
    "traceroute": "Reconnaissance",
    "dns_scan": "Reconnaissance",
    "network_scan": "Reconnaissance",
    "service_scan": "Reconnaissance",
    # Web Enumeration
    "dirb": "Web Enumeration",
    "wpscan": "Web Enumeration",
    "attacker_http": "Web Enumeration",
    # Exploitation
    "webshell_upload": "Exploitation",
    "webshell_cmd": "Exploitation",
    # Password Cracking
    "crack_passwords": "Password Cracking",
    # Privilege Escalation
    "escalate": "Privilege Escalation",
    "attacker_change_user": "Privilege Escalation",
    "escalated_command": "Privilege Escalation",
    "escalated_sudo_command": "Privilege Escalation",
    "escalated_sudo_session": "Privilege Escalation",
    # Exfiltration
    "dnsteal": "Exfiltration",
    "dnsteal-received": "Exfiltration",
    "dnsteal-dropped": "Exfiltration",
    "exfiltration-service": "Exfiltration",
    "attacker": "Exfiltration",
}

# Verify all labels are mapped
unmapped = set(global_label_ctr.keys()) - set(LABEL_TO_PHASE.keys())
if unmapped:
    print(f"WARNING: Unmapped labels: {unmapped}")
else:
    print("All 22 labels mapped to attack phases.")

print()

# Phase summary
phase_counts = Counter()
for lbl_name, cnt in global_label_ctr.items():
    phase_counts[LABEL_TO_PHASE[lbl_name]] += cnt

print(f"{'Phase':<25} {'Label Occurrences':>20}")
print("-" * 47)
for phase, cnt in phase_counts.most_common():
    labels_in_phase = [lbl for lbl, ph in LABEL_TO_PHASE.items() if ph == phase]
    print(f"{phase:<25} {cnt:>20,}")
    for lbl_name in sorted(labels_in_phase):
        print(f"  {lbl_name:<23} {global_label_ctr[lbl_name]:>20,}")

All 22 labels mapped to attack phases.

Phase                        Label Occurrences
-----------------------------------------------
Exfiltration                           159,168
  attacker                              53,056
  dnsteal                               53,056
  dnsteal-dropped                           48
  dnsteal-received                      53,006
  exfiltration-service                       2
Web Enumeration                         15,423
  attacker_http                          7,723
  dirb                                   4,493
  wpscan                                 3,207
Initial Access                           8,752
  attacker_vpn                              28
  foothold                               8,724
Reconnaissance                             965
  dns_scan                                 414
  network_scan                              92
  service_scan                             455
  traceroute                                 4
Privilege Escalatio

### 4.4 Rule Inventory

In [18]:
# Global rule frequency
global_rule_ctr = Counter()
for rules_dict in df_all["rules"]:
    for rule_list in rules_dict.values():
        for rule_name in rule_list:
            global_rule_ctr[rule_name] += 1

print(f"=== Global Rule Frequency ({len(global_rule_ctr)} unique rules) ===")
print(f"{'Rule':<55} {'Count':>8}")
print("-" * 65)
for rule_name, cnt in global_rule_ctr.most_common():
    print(f"{rule_name:<55} {cnt:>8,}")

=== Global Rule Frequency (36 unique rules) ===
Rule                                                       Count
-----------------------------------------------------------------
dnsteal.domain.match                                     106,108
dnsteal.domain.received                                   53,006
attacker.foothold.apache.access                           15,374
attacker.dirb.time                                         4,485
attacker.wpscan.time                                       3,199
attacker_foothold_dnsmasq_service_scan_sub_query             654
attacker_foothold_dnsmasq_dns_brute_scan_sub_query           558
attacker_foothold_dnsmasq_dns_brute_scan_parent_query        270
attacker_foothold_dnsmasq_service_scan_parent_query          232
attacker_foothold_dnsmasq_nmap_scan_sub_query                136
attacker.escalate.wpcrack                                     98
attacker.foothold.apache.access_error                         96
attacker.foothold.vpn.ip                 

In [19]:
# Rules-to-labels mapping: which rules trigger which labels
rule_to_labels = {}
for _, row in df_all.iterrows():
    for lbl_name, rule_list in row["rules"].items():
        for rule_name in rule_list:
            if rule_name not in rule_to_labels:
                rule_to_labels[rule_name] = set()
            rule_to_labels[rule_name].add(lbl_name)

print(f"=== Rule -> Label Mapping ({len(rule_to_labels)} rules) ===")
for rule_name in sorted(rule_to_labels.keys()):
    labels = sorted(rule_to_labels[rule_name])
    print(f"  {rule_name}")
    print(f"    -> {', '.join(labels)}")

=== Rule -> Label Mapping (36 rules) ===
  attacker.dirb.time
    -> dirb
  attacker.escalate.audit.su.login
    -> attacker_change_user, escalate
  attacker.escalate.audit.sudo.command.events
    -> escalate, escalated_command, escalated_sudo_command
  attacker.escalate.audit.sudo.command.start
    -> escalate, escalated_command, escalated_sudo_command
  attacker.escalate.su.login
    -> attacker_change_user, escalate
  attacker.escalate.sudo.command
    -> escalate, escalated_command, escalated_sudo_command
  attacker.escalate.sudo.open
    -> escalate, escalated_command, escalated_sudo_command, escalated_sudo_session
  attacker.escalate.systemd.newsession.after
    -> attacker_change_user, escalate
  attacker.escalate.webshell.cmd.http
    -> webshell_cmd
  attacker.escalate.webshell.cmd.http_prepare_crack
    -> escalate, webshell_cmd
  attacker.escalate.wpcrack
    -> crack_passwords, escalate
  attacker.foothold.apache.access
    -> attacker_http, foothold
  attacker.foothold.apa

### 4.5 Coverage Summary

In [20]:
# Coverage: labeled lines vs total raw lines per file
print(f"{'Host':<20} {'Log':<15} {'Labeled':>8} {'Raw Total':>10} {'Coverage':>9}")
print("-" * 65)
total_labeled = 0
total_raw = 0
for entry in LABEL_FILES:
    df_file = df_all[
        (df_all["source_host"] == entry["host"]) & (df_all["source_log"] == entry["log"])
    ]
    n_labeled = len(df_file)
    n_raw = entry["raw_lines"]
    pct = n_labeled / n_raw * 100
    total_labeled += n_labeled
    total_raw += n_raw
    print(f"{entry['host']:<20} {entry['log']:<15} {n_labeled:>8,} {n_raw:>10,} {pct:>8.1f}%")

print("-" * 65)
print(
    f"{'TOTAL':<20} {'':<15} {total_labeled:>8,} {total_raw:>10,} {total_labeled / total_raw * 100:>8.1f}%"
)

Host                 Log              Labeled  Raw Total  Coverage
-----------------------------------------------------------------
inet-firewall        dnsmasq.log       54,035    275,900     19.6%
intranet_server      access.log.2       7,695      8,530     90.2%
intranet_server      error.log.2           36         36    100.0%
intranet_server      audit.log              9      2,316      0.4%
intranet_server      auth.log               8        272      2.9%
monitoring           cpu.log               49      1,920      2.6%
vpn                  openvpn.log           28      5,537      0.5%
internal_share       audit.log              2        732      0.3%
-----------------------------------------------------------------
TOTAL                                  61,862    295,243     21.0%


## 5. Raw 1:1 DataFrame (`df_raw`)

Construct `df_raw` matching the staging DDL. Labels and rules are stored as JSON text blobs (not parsed), preserving the 1:1 mapping between JSONL records and database rows.

In [21]:
# Build df_raw with TEXT blob columns for labels and rules
df_raw = pd.DataFrame(
    {
        "source_host": df_all["source_host"],
        "source_log": df_all["source_log"],
        "line": df_all["line"],
        "labels_json": df_all["labels"].apply(json.dumps),
        "rules_json": df_all["rules"].apply(json.dumps),
    }
)

print(f"df_raw shape: {df_raw.shape}")
print(f"Columns: {list(df_raw.columns)}")
print()
print("Data types:")
print(df_raw.dtypes.to_string())
print()
print("Null counts:")
print(df_raw.isnull().sum().to_string())
print()
print("=== First 5 rows ===")
print(df_raw.head().to_string())
print()
print("=== Last 5 rows ===")
print(df_raw.tail().to_string())

df_raw shape: (61862, 5)
Columns: ['source_host', 'source_log', 'line', 'labels_json', 'rules_json']

Data types:
source_host    object
source_log     object
line            int64
labels_json    object
rules_json     object

Null counts:
source_host    0
source_log     0
line           0
labels_json    0
rules_json     0

=== First 5 rows ===
     source_host   source_log  line                                  labels_json                                                                                                                    rules_json
0  inet-firewall  dnsmasq.log     1  ["dnsteal", "attacker", "dnsteal-received"]  {"dnsteal": ["dnsteal.domain.match"], "attacker": ["dnsteal.domain.match"], "dnsteal-received": ["dnsteal.domain.received"]}
1  inet-firewall  dnsmasq.log     2  ["dnsteal", "attacker", "dnsteal-received"]  {"dnsteal": ["dnsteal.domain.match"], "attacker": ["dnsteal.domain.match"], "dnsteal-received": ["dnsteal.domain.received"]}
2  inet-firewall  dnsmasq.log     

In [22]:
# Verify df_raw integrity
assert len(df_raw) == 61_862, f"Expected 61,862 rows, got {len(df_raw)}"
assert list(df_raw.columns) == ["source_host", "source_log", "line", "labels_json", "rules_json"]
assert df_raw.isnull().sum().sum() == 0, "Unexpected nulls in df_raw"

# Verify JSON round-trip
for idx in [0, len(df_raw) // 2, len(df_raw) - 1]:
    parsed_labels = json.loads(df_raw.iloc[idx]["labels_json"])
    parsed_rules = json.loads(df_raw.iloc[idx]["rules_json"])
    assert isinstance(parsed_labels, list)
    assert isinstance(parsed_rules, dict)

print("df_raw integrity checks passed:")
print(f"  Rows: {len(df_raw):,}")
print(f"  Columns: {len(df_raw.columns)}")
print("  Nulls: 0")
print("  JSON round-trip: OK")

df_raw integrity checks passed:
  Rows: 61,862
  Columns: 5
  Nulls: 0
  JSON round-trip: OK


In [23]:
# The staging candidate key is (source_host, source_log, line_number) for this dataset
composite_key_dupes = df_raw.duplicated(
    subset=["source_host", "source_log", "line"], keep=False
).sum()
print(f"Duplicate (source_host, source_log, line) rows: {composite_key_dupes}")
assert composite_key_dupes == 0, f"Expected 0 duplicates, got {composite_key_dupes}"
print(f"Candidate key (source_host, source_log, line) verified: {len(df_raw):,} unique rows")

Duplicate (source_host, source_log, line) rows: 0
Candidate key (source_host, source_log, line) verified: 61,862 unique rows


## 6. Schema Mapping

Map the raw fields to the planned `stg_attack_label_line_raw` table.

### 6.0 Type inference assumptions

The label files are structured JSONL (not raw unstructured text), so type inference is straightforward. See `type_inference_assumptions.md` (at data-201/ root) for the shared team-level rules.

Key decisions for this file:
- `source_host` and `source_log`: VARCHAR with lengths based on observed max values
- `line`: INTEGER (max observed: 254,393 in dnsmasq labels; fits INTEGER)
- `labels_json` and `rules_json`: TEXT blobs storing the original JSON arrays/objects

### 6.1 Column Mapping

| # | Raw Field | DB Column | PostgreSQL Type | MySQL Type | Nullable | Notes |
|---|-----------|-----------|-----------------|------------|----------|-------|
| 1 | (derived) | `row_id` | `SERIAL` | `INT AUTO_INCREMENT` | NOT NULL | Surrogate PK |
| 2 | source_host | `source_host` | `VARCHAR(20)` | `VARCHAR(20)` | NOT NULL | Host name (max 16 chars observed) |
| 3 | source_log | `source_log` | `VARCHAR(20)` | `VARCHAR(20)` | NOT NULL | Log file name (max 13 chars observed) |
| 4 | line | `line_number` | `INTEGER` | `INT` | NOT NULL | Line in raw log file (max 254,393) |
| 5 | labels | `labels_json` | `TEXT` | `TEXT` | NOT NULL | JSON array of label strings (1NF violation) |
| 6 | rules | `rules_json` | `TEXT` | `TEXT` | NOT NULL | JSON object mapping labels to rule arrays (1NF violation) |

### 6.2 Staging DDL

In [27]:
postgresql_ddl = """
-- PostgreSQL
CREATE TABLE stg_attack_label_line_raw (
    row_id          SERIAL PRIMARY KEY,
    source_host     VARCHAR(20) NOT NULL,
    source_log      VARCHAR(20) NOT NULL,
    line_number     INTEGER NOT NULL,
    labels_json     TEXT NOT NULL,
    rules_json      TEXT NOT NULL
);
"""

mysql_ddl = """
-- MySQL
CREATE TABLE stg_attack_label_line_raw (
    row_id          INT AUTO_INCREMENT PRIMARY KEY,
    source_host     VARCHAR(20) NOT NULL,
    source_log      VARCHAR(20) NOT NULL,
    line_number     INT NOT NULL,
    labels_json     TEXT NOT NULL,
    rules_json      TEXT NOT NULL
);
"""

print(postgresql_ddl)
print(mysql_ddl)


-- PostgreSQL
CREATE TABLE stg_attack_label_line_raw (
    row_id          SERIAL PRIMARY KEY,
    source_host     VARCHAR(20) NOT NULL,
    source_log      VARCHAR(20) NOT NULL,
    line_number     INTEGER NOT NULL,
    labels_json     TEXT NOT NULL,
    rules_json      TEXT NOT NULL
);


-- MySQL
CREATE TABLE stg_attack_label_line_raw (
    row_id          INT AUTO_INCREMENT PRIMARY KEY,
    source_host     VARCHAR(20) NOT NULL,
    source_log      VARCHAR(20) NOT NULL,
    line_number     INT NOT NULL,
    labels_json     TEXT NOT NULL,
    rules_json      TEXT NOT NULL
);



## 7. Normalization Observations

Applying the `normalization_rules_sheet.md` checklist to the label data.

### 7.1 1NF Check

**Multi-valued field 1: `labels_json`**
Each row contains a JSON array of 2-4 label strings (e.g., `["attacker", "dnsteal", "dnsteal-received"]`). This is a 1NF violation-multiple distinct values stored in one cell.

Resolution: junction table `attack_label_entries(label_entry_id, label_record_id, label_name)` with one row per label per record.

**Multi-valued field 2: `rules_json`**
Each row contains a nested JSON object mapping labels to arrays of rule names (e.g., `{"dnsteal": ["dnsteal.domain.match"], "attacker": ["dnsteal.domain.match"]}`). This is a 1NF violation-nested multi-valued structure.

Resolution: junction table `attack_label_rules(rule_entry_id, label_record_id, label_name, rule_name)` with one row per rule per label per record.

**1NF status: violated.** Both `labels_json` and `rules_json` contain multi-valued data.

### 7.2 2NF Check

The raw table uses a single-column surrogate primary key (`row_id`). Partial dependencies only arise with composite keys.

The staging candidate key is `(source_host, source_log, line_number)` for this dataset, unique-verified in Section 5 (0 duplicates across 61,862 rows). Each line in a given file appears at most once. If this composite key were chosen as PK, both `labels_json` and `rules_json` depend on the full composite (the labels for line 1860 in intranet_server/audit.log are different from any line 1860 in another file), so no partial dependency exists.

**2NF status: satisfied** (with either surrogate or composite key).

### 7.3 3NF Check

**Observations from the data:**

The `labels` field contains label name strings (e.g., `dnsteal`, `escalate`). Each of the 22 observed label names maps to exactly one of 7 attack phases in the project taxonomy (`data_scope_and_findings.md`)-verified deterministic in Section 4.3 (22 labels, 0 inconsistencies). This is the FD `label_name -> attack_phase`.

However, `attack_phase` does not exist anywhere in the JSONL data. It is external domain knowledge. This FD is relevant for staging/ETL design: if a future schema includes both `label_name` and `attack_phase` as columns, the transitive dependency `PK -> label_name -> attack_phase` would be a 3NF violation, resolvable by extracting a lookup table.

**Does `rule_name -> label_name`?** No. 29 of 36 rules trigger multiple labels (verified in Section 7 cell output). The rule-to-label relationship is many-to-many.

**3NF observation:** No transitive dependency exists within the JSONL data fields themselves (`line`, `labels`, `rules`). The `label_name -> attack_phase` FD is external taxonomy and relevant for normalization planning.

In [25]:
# Verify: label_name -> attack_phase is a valid FD
# Each label should map to exactly one phase
label_phase_check = {}
for lbl_name, phase in LABEL_TO_PHASE.items():
    if lbl_name in label_phase_check:
        if label_phase_check[lbl_name] != phase:
            print(f"  VIOLATION: {lbl_name} maps to both {label_phase_check[lbl_name]} and {phase}")
    label_phase_check[lbl_name] = phase

print(f"label_name -> attack_phase FD verified: {len(label_phase_check)} labels, all deterministic")
print()

# Verify: rule_name does NOT determine label_name
rule_label_sets = {}
for _, row in df_all.iterrows():
    for lbl_name, rule_list in row["rules"].items():
        for rule_name in rule_list:
            if rule_name not in rule_label_sets:
                rule_label_sets[rule_name] = set()
            rule_label_sets[rule_name].add(lbl_name)

multi_label_rules = {r: lbls for r, lbls in rule_label_sets.items() if len(lbls) > 1}
print(f"Rules that trigger multiple labels: {len(multi_label_rules)}")
for rule_name, lbls in sorted(multi_label_rules.items()):
    print(f"  {rule_name} -> {sorted(lbls)}")
print()
print("rule_name -> label_name is NOT an FD (many-to-many relationship)")

label_name -> attack_phase FD verified: 22 labels, all deterministic

Rules that trigger multiple labels: 29
  attacker.escalate.audit.su.login -> ['attacker_change_user', 'escalate']
  attacker.escalate.audit.sudo.command.events -> ['escalate', 'escalated_command', 'escalated_sudo_command']
  attacker.escalate.audit.sudo.command.start -> ['escalate', 'escalated_command', 'escalated_sudo_command']
  attacker.escalate.su.login -> ['attacker_change_user', 'escalate']
  attacker.escalate.sudo.command -> ['escalate', 'escalated_command', 'escalated_sudo_command']
  attacker.escalate.sudo.open -> ['escalate', 'escalated_command', 'escalated_sudo_command', 'escalated_sudo_session']
  attacker.escalate.systemd.newsession.after -> ['attacker_change_user', 'escalate']
  attacker.escalate.webshell.cmd.http_prepare_crack -> ['escalate', 'webshell_cmd']
  attacker.escalate.wpcrack -> ['crack_passwords', 'escalate']
  attacker.foothold.apache.access -> ['attacker_http', 'foothold']
  attacker.footh

### 7.4 Preliminary Functional Dependencies

Observed from the data (no table exists yet-these inform staging/ETL design):

| FD | Determinant | Dependent(s) | Reasoning |
|---|---|---|---|
| FD1 | `(source_host, source_log, line)` | `labels`, `rules` | Each line in a given file has exactly one label record. Verified: 0 duplicates across 61,862 rows. |
| FD2 | `labels` array | `rules` dict keys | Structural constraint: rules keys always equal labels entries (0 mismatches). Not a traditional FD-an integrity constraint in the source data. |

**Post-decomposition FD (for normalization planning):**

| FD | Determinant | Dependent(s) | Source | Reasoning |
|---|---|---|---|---|
| FD3 | `label_name` | `attack_phase` | Project taxonomy (`data_scope_and_findings.md`) | Each of 22 labels maps to exactly 1 of 7 phases. Verified deterministic. Not present in JSONL data-external domain knowledge. |

## 8. Summary

In [26]:
print("=== Summary ===")
print(f"Total label records:     {len(df_raw):,}")
print("Source files:            8 (across 5 hosts)")
print(f"Unique labels:           {len(global_label_ctr)}")
print(f"Unique label combos:     {len(global_combo_ctr)}")
print(f"Unique rules:            {len(global_rule_ctr)}")
print(f"Columns in df_raw:       {len(df_raw.columns)}")
print(
    f"Overall coverage:        {len(df_raw):,} / {sum(e['raw_lines'] for e in LABEL_FILES):,} "
    f"= {len(df_raw) / sum(e['raw_lines'] for e in LABEL_FILES) * 100:.1f}% of raw log lines are labeled"
)
print()
print("=== Normalization Observations (data structure, no tables exist yet) ===")
print(
    "  1NF: labels field is multi-valued (2-4 strings per row); rules field is nested multi-valued dict"
)
print("  2NF: (source_host, source_log, line) is a valid candidate key (0 duplicates verified)")
print("  3NF: No transitive dependency within the JSONL fields themselves")
print(
    "       label_name -> attack_phase FD exists in project taxonomy (external, not in JSONL data)"
)
print("       rule_name -> label_name is NOT an FD (29/36 rules trigger multiple labels)")
print()
print("=== Relational Importance ===")
print(
    "Labels bridge raw log events and annotation semantics via (source_host, source_log, line_number)."
)
print("Enables: label timeline reconstruction, labeled event filtering, coverage queries")

=== Summary ===
Total label records:     61,862
Source files:            8 (across 5 hosts)
Unique labels:           22
Unique label combos:     21
Unique rules:            36
Columns in df_raw:       5
Overall coverage:        61,862 / 295,243 = 21.0% of raw log lines are labeled

=== Normalization Observations (data structure, no tables exist yet) ===
  1NF: labels field is multi-valued (2-4 strings per row); rules field is nested multi-valued dict
  2NF: (source_host, source_log, line) is a valid candidate key (0 duplicates verified)
  3NF: No transitive dependency within the JSONL fields themselves
       label_name -> attack_phase FD exists in project taxonomy (external, not in JSONL data)
       rule_name -> label_name is NOT an FD (29/36 rules trigger multiple labels)

=== Relational Importance ===
Labels bridge raw log events and annotation semantics via (source_host, source_log, line_number).
Enables: label timeline reconstruction, labeled event filtering, coverage queries
